In [ ]:
# Import functions from preprocessing.py
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.optim as optim
from tqdm import tqdm

from GPModel import GPModel
from GPArealModel import GPArealModel
from VIGP_Unlinked import VIGP_Unlinked


# Add the path to the src directory
sys.path.append(os.path.abspath(os.path.join('..', 'data')))

result = {}
B = 169
n_i = 6
seed = 4
input_dim = 1
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

niter_GP=10
niter_GPAreal=10
niter_VI= 50

# Load data from the specified path
data_path = os.path.join('..', 'data','beta', f'B_{B}_n_{n_i}', f'data_seed_{seed}.pt')
data = torch.load(data_path)

# Extract variables from the data dictionary
y = data['y']
region_assignments = data['region_assignments']
x = data['x']
w = data['w']
e = data['e']
s = data['s']
x_jumbled_within_regions = data['x_jumbled_within_regions']
s_jumbled_within_regions = data['s_jumbled_within_regions']
perm_matrix_x = data['perm_matrix_x']
perm_matrix_s = data['perm_matrix_s']
sigmasq_true = data['sigmasq_true']
phi_true = data['phi_true']
beta_true = data['beta_true']
nu_true = data['nu_true']
tausq_true = data['tausq_true']


# Train the GPmodel (oracle)
# Oracle GP model if the locations and links are known
# Initialize and optimize the model
model = GPModel().to(device)
optimizer = optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.01)

for i in tqdm(range(niter_GP)):
    optimizer.zero_grad()
    loss = model(s, x, y)
    loss.backward()
    optimizer.step()
    
    # Constrain sigmasq, length_scale, and tausq to be positive
    with torch.no_grad():
        model.sigmasq.clamp_(min=1e-6)
        model.phi.clamp_(min=1e-6)
        model.tausq.clamp_(min=1e-6)
        
# Save the model parameters
model_params = {
    'nu': model.nu.item(),
    'phi': model.phi.item(),
    'sigmasq': model.sigmasq.item(),
    'tausq': model.tausq.item(),
    'beta': model.beta.detach().cpu().numpy()
}

# Save the model parameters to a file
result['GPmodel'] = model_params


# Train the model GPareal
# Compute region-wise averages directly
unique_regions = torch.unique(region_assignments)
B = len(unique_regions)

ybar = torch.zeros(B, device=y.device)
xbar = torch.zeros(B, input_dim, device=x.device)

for i, region in enumerate(unique_regions):
    # Get indices for the current region
    indices = torch.where(region_assignments == region)[0]
    
    # Compute region-wise averages for y and x
    ybar[i] = torch.mean(y[indices])
    xbar[i] = torch.mean(x_jumbled_within_regions[indices], dim=0)


# Initialize and optimize the model
model = GPArealModel().to(device)
optimizer = optim.AdamW(model.parameters(), lr=0.01, weight_decay=0.01)

for i in tqdm(range(niter_GPAreal)):
    optimizer.zero_grad()
    loss = model(s_jumbled_within_regions, region_assignments, xbar, ybar)
    loss.backward()
    optimizer.step()


# Save the model parameters
model_params = {
    'nu': model.nu.item(),
    'phi': model.phi.item(),
    'sigmasq': model.sigmasq.item(),
    'tausq': model.tausq.item(),
    'beta': model.beta.detach().cpu().numpy()
}
# Save the model parameters to a file
result['GPArealModel'] = model_params


# Train the model VIGP_unlinked
n_blocks = B
n_locations = n_i

# Random toy data for X and Y
X = torch.tensor(x_jumbled_within_regions, dtype=torch.float32).reshape(n_blocks, n_locations)
Y = torch.tensor(y, dtype=torch.float32).reshape(n_blocks, n_locations)

# Generate (n_blocks * n_locations) 2D coordinates
total_points = n_blocks * n_locations
locations = torch.tensor(s_jumbled_within_regions,dtype=torch.float32)

# Compute distance matrix from locations
Dist = torch.cdist(locations, locations, p=2)  # Pairwise distances
Dist = (Dist + Dist.T) / 2  # Make it symmetric because numerical errors can cause asymmetry

# Set optional args
n_steps = 50
n_phi_samples = 100
n_piX_sample = 50
tau_X = 0.8
tau_S = 0.8
n_piS_sample = 50

#informative prior
# prior_parameters = {
#     "a1": 490,
#     "b1": (490-1)*result['GPArealModel']['sigmasq'],
#     "a2": 490,  # Using the previous entry
#     "b2": (490-1)*result['GPArealModel']['tausq'],
#     "eta_X_sq": 0.1,
#     "eta_S_sq": 0.1,
#     "mu_beta": result['GPArealModel']['beta'][0],
#     "sigmasq_beta": 1,
#     "phi_prior_ub": torch.max(torch.tensor([1/torch.max(Dist), result['GPArealModel']['phi']-0.5])),
#     "phi_prior_lb": result['GPArealModel']['phi'] + 0.5
# }

#uninformative prior
prior_parameters = {
    "a1": 0.1,
    "b1": 0.1,
    "a2": 0.1,  # Using the previous entry
    "b2": 0.1,
    "eta_X_sq": 0.1,
    "eta_S_sq": 0.1,
    "mu_beta": 0,
    "sigmasq_beta": 100,
    "phi_prior_lb": torch.max(Dist)*0.01,
    "phi_prior_ub":torch.max(Dist)
}


for tau in [0.3]:
    tau_X = tau
    tau_S = tau

    results_VI = VIGP_Unlinked(
        n_iter=niter_VI,
        n_blocks=n_blocks,
        n_locations=n_locations,
        X=X,
        Y=Y,
        Dist=Dist,
        n_steps=n_steps,
        n_phi_samples=n_phi_samples,
        n_piX_sample=n_piX_sample,
        tau_X=tau_X, tau_S=tau_S,
        n_piS_sample=n_piS_sample,
        seed=521, 
        fix_piX= False, 
        fix_piS= False,
        fix_mu_lambda_beta=False,
        fix_sigmasq_lambda_beta=False,
        fix_lambda_b1=False,
        lambda_b1_fixed=((B*n_i)*0.5 + 0.1) * 5,
        fix_lambda_b2=False,
        M_X_star_fixed=perm_matrix_x.T,
        M_S_star_fixed=perm_matrix_s.T,
        V_X_star_fixed=torch.eye(n_locations, n_locations, device=device),
        V_S_star_fixed=torch.eye(n_locations, n_locations, device=device), 
        phi_init = 0.5,
        mean_Rphi_inv_fixed= torch.linalg.inv(torch.exp(-4 * Dist)),
        fix_mean_Rphi_inv=False, 
        pi_X_true = perm_matrix_x.T,
        pi_S_true = perm_matrix_s.T,
        VX_ub = 0.5,
        VS_ub=0.5,
        lr_piX = 0.01,
        lr_piS = 0.01, 
        prior_parameters = prior_parameters
    )
   
    # Save the model parameters to the result dictionary
    result[f'VIGP_unlinked_tau_{tau}'] = results_VI

result0 = result.copy()
# Save the result dictionary to a file
# result_path = os.path.join('..', 'data', 'results' , f'B_{B}_n_{n_i}', f'results_seed_{seed}.pt')
# os.makedirs(os.path.dirname(result_path), exist_ok=True)
# torch.save(result, result_path)

/var/folders/w7/jxz2zn316391355qstwl03940000gn/T/ipykernel_80448/3425257170.py:32: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(data_path)
  0%|          

Minimum eigenvalue of V_X_star - M_X_star.T @ M_X_star: -1.3007e-07
Minimum eigenvalue of V_S_star - M_S_star.T @ M_S_star: -1.3007e-07


  2%|▏         | 1/50 [00:40<32:56, 40.33s/it]

Iter 1/50 | mu_lambda_beta: 5.7485 | 
 sigmasq_lambda_beta: 0.0000 | 
 lambda_a1: 507.1000 | lambda_b1: 1486.5884 | lambda_a2: 507.1000 | lambda_b2: 7286.8350
‣  E[ϕ]: 0.4510 | ‣ ||mu_W||: 21.9601
Number of correct permutations recognized for piX: 6.0
Number of correct permutations recognized for piS: 4.0
Total Loss: 3.3084
Minimum eigenvalue of V_X_star - M_X_star.T @ M_X_star: 2.8795e-02
Minimum eigenvalue of V_S_star - M_S_star.T @ M_S_star: 4.3363e-02


  4%|▍         | 2/50 [01:22<32:59, 41.23s/it]

...

Iter 16/50 | mu_lambda_beta: 7.8550 | 
 sigmasq_lambda_beta: 0.0149 | 
 lambda_a1: 507.1000 | lambda_b1: 567.0295 | lambda_a2: 507.1000 | lambda_b2: 2429.5894
‣  E[ϕ]: 0.7524 | ‣ ||mu_W||: 18.1665
Number of correct permutations recognized for piX: 6.0
Number of correct permutations recognized for piS: 6.0
Total Loss: 2.1922
Minimum eigenvalue of V_X_star - M_X_star.T @ M_X_star: 7.7962e-03
Minimum eigenvalue of V_S_star - M_S_star.T @ M_S_star: 1.0452e-01


 34%|███▍      | 17/50 [12:08<24:10, 43.94s/it]

Iter 17/50 | mu_lambda_beta: 7.8498 | 
 sigmasq_lambda_beta: 0.0149 | 
 lambda_a1: 507.1000 | lambda_b1: 545.8812 | lambda_a2: 507.1000 | lambda_b2: 2436.6006
‣  E[ϕ]: 0.7522 | ‣ ||mu_W||: 17.7957
Number of correct permutations recognized for piX: 6.0
Number of correct permutations recognized for piS: 6.0
Total Loss: 2.1956
Minimum eigenvalue of V_X_star - M_X_star.T @ M_X_star: 7.7996e-03
Minimum eigenvalue of V_S_star - M_S_star.T @ M_S_star: 1.0766e-01


 34%|███▍      | 17/50 [12:29<24:15, 44.09s/it]


KeyboardInterrupt: 

In [1]:
# analysis.py — tailored to your vary_B data layout
import sys
import os
import re
from tqdm import tqdm
import torch
import torch.optim as optim
import numpy as np

from GPModel import GPModel
from GPArealModel import GPArealModel
from VIGP_Unlinked import VIGP_Unlinked

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
input_dim = 1

# --------- CLI ---------
# Usage:
#   python analysis.py B n_i seed
#   python analysis.py B n_i seed phi snr
# #if len(sys.argv) < 4:
#     raise ValueError("Usage: python analysis.py B n_i seed [phi snr]")

B_arg   = 49
n_i_arg = 6
seed    = 2

phi_cli = None
snr_cli = None
# if len(sys.argv) >= 6:
#     phi_cli = float(sys.argv[4])
#     snr_cli = float(sys.argv[5])

# --------- Resolve data path ---------
base_dir = os.path.join('..', 'data', 'vary_B', f'B_{B_arg}_n_{n_i_arg}')

def autodetect_phi_snr(folder):
    if not os.path.isdir(folder):
        raise FileNotFoundError(f"Folder not found: {folder}")
    candidates = [d for d in os.listdir(folder) if os.path.isdir(os.path.join(folder, d)) and d.startswith('phi_')]
    if len(candidates) == 0:
        raise FileNotFoundError(f"No phi/snr subfolders found in {folder}")
    if len(candidates) > 1:
        # Try to pick unique; otherwise ask user to pass explicitly
        raise ValueError(f"Multiple phi/snr folders found in {folder}: {candidates}. "
                         f"Re-run with explicit phi and snr.")
    d = candidates[0]  # e.g., 'phi_2.0_snr_1.0e+00'
    m = re.match(r'^phi_([^_]+)_snr_([^/]+)$', d)
    if not m:
        raise ValueError(f"Cannot parse phi/snr from folder name: {d}")
    return float(m.group(1)), float(m.group(2)), d

if phi_cli is None or snr_cli is None:
    phi_resolved, snr_resolved, phi_snr_dir = autodetect_phi_snr(base_dir)
else:
    phi_resolved, snr_resolved = phi_cli, snr_cli
    phi_snr_dir = f"phi_{phi_resolved}_snr_{snr_resolved:.1e}"

data_path = os.path.join(base_dir, phi_snr_dir, f"data_seed_{seed}.pt")

# --------- Load data ---------
data = torch.load(data_path, map_location=device)

y  = data['y'].to(device).float()
x  = data['x'].to(device).float()
w  = data['w'].to(device).float()
e  = data['e'].to(device).float()
s  = data['s'].to(device).float()
region_assignments = data['region_assignments'].to(device).long()

x_jumbled_within_regions = data['x_jumbled_within_regions'].to(device).float()
s_jumbled_within_regions = data['s_jumbled_within_regions'].to(device).float()

perm_matrix_x = data['perm_matrix_x'].to(device).float()
perm_matrix_s = data['perm_matrix_s'].to(device).float()

sigmasq_true = float(data['sigmasq_true'])
phi_true     = float(data['phi_true'])
beta_true    = float(data['beta_true'])
nu_true      = float(data['nu_true'])
tausq_true   = float(data['tausq_true'])
snr_true     = float(data.get('snr', snr_resolved))

# Sanity: unique region count
unique_regions = torch.unique(region_assignments)
B_in_data = len(unique_regions)
if B_in_data != B_arg:
    print(f"[WARN] B in data ({B_in_data}) != B from CLI ({B_arg}). Using B_in_data for shapes.")
n_blocks = B_in_data
n_locations = n_i_arg  # expected design; will assert below

N = y.numel()
if N % n_blocks != 0:
    raise ValueError(f"N={N} not divisible by B={n_blocks}")
if n_locations != (N // n_blocks):
    print(f"[WARN] n_i from CLI ({n_i_arg}) != inferred ({N // n_blocks}). Using inferred.")
    n_locations = N // n_blocks

# --------- Training iters ---------
niter_GP = 3000
niter_GPAreal = 3000
niter_VI = 100

result = {}
torch.manual_seed(521)

# ===================== 1) Oracle GP (locations & links known) =====================
gp = GPModel().to(device)
opt = optim.AdamW(gp.parameters(), lr=0.01, weight_decay=0.01)

for _ in tqdm(range(niter_GP), desc="Train GPModel (oracle)"):
    opt.zero_grad()
    loss = gp(s, x, y)
    loss.backward()
    opt.step()
    with torch.no_grad():
        gp.sigmasq.clamp_(min=1e-6)
        gp.phi.clamp_(min=1e-6)
        gp.tausq.clamp_(min=1e-6)

result['GPmodel'] = {
    'nu': float(gp.nu.item()),
    'phi': float(gp.phi.item()),
    'sigmasq': float(gp.sigmasq.item()),
    'tausq': float(gp.tausq.item()),
    'beta': gp.beta.detach().float().cpu().numpy(),
    'true_params': {
        'nu_true': nu_true, 'phi_true': phi_true, 'sigmasq_true': sigmasq_true,
        'tausq_true': tausq_true, 'beta_true': beta_true, 'snr_true': snr_true
    }
}

# ===================== 2) Areal GP (region-averaged) =====================
# Region-wise averages
ybar = torch.zeros(n_blocks, device=device)
xbar = torch.zeros(n_blocks, input_dim, device=device)
for i, region in enumerate(unique_regions):
    idx = torch.where(region_assignments == region)[0]
    ybar[i] = torch.mean(y[idx])
    xbar[i] = torch.mean(x_jumbled_within_regions[idx], dim=0)

gpa = GPArealModel().to(device)
opt = optim.AdamW(gpa.parameters(), lr=0.01, weight_decay=0.01)
for _ in tqdm(range(niter_GPAreal), desc="Train GPArealModel"):
    opt.zero_grad()
    loss = gpa(s_jumbled_within_regions, region_assignments, xbar, ybar)
    loss.backward()
    opt.step()
    with torch.no_grad():
        gpa.sigmasq.clamp_(min=1e-6)
        gpa.phi.clamp_(min=1e-6)
        gpa.tausq.clamp_(min=1e-6)

result['GPArealModel'] = {
    'nu': float(gpa.nu.item()),
    'phi': float(gpa.phi.item()),
    'sigmasq': float(gpa.sigmasq.item()),
    'tausq': float(gpa.tausq.item()),
    'beta': gpa.beta.detach().float().cpu().numpy()
}

# ===================== 3) VI for Unlinked GP =====================
# Reshape to (B, n_i)
X = x_jumbled_within_regions.reshape(n_blocks, n_locations).contiguous()
Y = y.reshape(n_blocks, n_locations).contiguous()

locations = s_jumbled_within_regions  # (N, d)
Dist = torch.cdist(locations, locations, p=2)
Dist = (Dist + Dist.T) / 2

n_steps = 50
n_phi_samples = 100
n_piX_sample = 50
n_piS_sample = 50

prior_parameters = {
    "a1": 0.1, "b1": 0.1,
    "a2": 0.1, "b2": 0.1,
    "eta_X_sq": 0.1, "eta_S_sq": 0.1,
    "mu_beta": 0.0, "sigmasq_beta": 100.0,
    "phi_prior_lb": torch.max(Dist)*0.01,
    "phi_prior_ub":torch.max(Dist)
}

for tau in [0.2, 0.4, 0.6, 0.8, 0.9]:
    results_VI = VIGP_Unlinked(
        n_iter=niter_VI,
        n_blocks=n_blocks,
        n_locations=n_locations,
        X=X, Y=Y,
        Dist=Dist,
        n_steps=n_steps,
        n_phi_samples=n_phi_samples,
        n_piX_sample=n_piX_sample,
        tau_X=tau, tau_S=tau,
        n_piS_sample=n_piS_sample,
        seed=521,
        fix_piX=False, fix_piS=False,
        fix_mu_lambda_beta=False,
        fix_sigmasq_lambda_beta=False,
        fix_lambda_b1=False,
        lambda_b1_fixed=((n_blocks * n_locations) * 0.5 + 0.1) * 5,
        fix_lambda_b2=False,
        M_X_star_fixed=perm_matrix_x.T,
        M_S_star_fixed=perm_matrix_s.T,
        V_X_star_fixed=torch.eye(n_locations, n_locations, device=device),
        V_S_star_fixed=torch.eye(n_locations, n_locations, device=device),
        phi_init=0.5,
        mean_Rphi_inv_fixed=torch.linalg.inv(torch.exp(-4 * Dist)),
        fix_mean_Rphi_inv=False,
        pi_X_true=perm_matrix_x.T,
        pi_S_true=perm_matrix_s.T,
        VX_ub=0.5, VS_ub=0.5,
        lr_piS=0.01, lr_piX=0.01,
        prior_parameters=prior_parameters
    )
    result[f'VIGP_unlinked_tau_{tau}'] = results_VI

# --------- Save results ---------
results_dir = os.path.join('..', 'data', 'results', 'vary_B',
                           f'B_{B_arg}_n_{n_i_arg}', phi_snr_dir)
os.makedirs(results_dir, exist_ok=True)
result_path = os.path.join(results_dir, f'results_seed_{seed}.pt')
torch.save(result, result_path)
print(f"[OK] Saved results to {result_path}")


/var/folders/w7/jxz2zn316391355qstwl03940000gn/T/ipykernel_80718/392791035.py:62: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(data_path, map_location=dev

Minimum eigenvalue of V_X_star - M_X_star.T @ M_X_star: -1.3007e-07
Minimum eigenvalue of V_S_star - M_S_star.T @ M_S_star: -1.3007e-07


  1%|          | 1/100 [00:22<37:52, 22.95s/it]

Iter 1/100 | mu_lambda_beta: 6.7472 | 
 sigmasq_lambda_beta: 0.0000 | 
 lambda_a1: 147.1000 | lambda_b1: 446.1736 | lambda_a2: 147.1000 | lambda_b2: 2106.6487
‣  E[ϕ]: 0.5692 | ‣ ||mu_W||: 16.6702
Number of correct permutations recognized for piX: 6.0
Number of correct permutations recognized for piS: 4.0
Total Loss: 3.2540
Minimum eigenvalue of V_X_star - M_X_star.T @ M_X_star: 1.9145e-02
Minimum eigenvalue of V_S_star - M_S_star.T @ M_S_star: 1.9449e-02


  2%|▏         | 2/100 [00:45<37:29, 22.96s/it]

...

Stopping early at step 1 due to minimal loss change.
Iter 84/100 | mu_lambda_beta: 7.4737 | 
 sigmasq_lambda_beta: 0.0566 | 
 lambda_a1: 147.1000 | lambda_b1: 115.1749 | lambda_a2: 147.1000 | lambda_b2: 805.9597
‣  E[ϕ]: 0.9768 | ‣ ||mu_W||: 15.2200
Number of correct permutations recognized for piX: 6.0
Number of correct permutations recognized for piS: 6.0
Total Loss: 2.3535
Minimum eigenvalue of V_X_star - M_X_star.T @ M_X_star: 2.7940e-02
Minimum eigenvalue of V_S_star - M_S_star.T @ M_S_star: 9.6372e-02
Stopping early at step 1 due to minimal loss change.


 85%|████████▌ | 85/100 [23:42<01:00,  4.03s/it]

Stopping early at step 1 due to minimal loss change.
Iter 85/100 | mu_lambda_beta: 7.4568 | 
 sigmasq_lambda_beta: 0.0569 | 
 lambda_a1: 147.1000 | lambda_b1: 115.1418 | lambda_a2: 147.1000 | lambda_b2: 814.3300
‣  E[ϕ]: 0.9768 | ‣ ||mu_W||: 15.2225
Number of correct permutations recognized for piX: 6.0
Number of correct permutations recognized for piS: 6.0
Total Loss: 2.3312
Minimum eigenvalue of V_X_star - M_X_star.T @ M_X_star: 2.7906e-02
Minimum eigenvalue of V_S_star - M_S_star.T @ M_S_star: 9.6373e-02


 85%|████████▌ | 85/100 [23:47<04:11, 16.79s/it]


KeyboardInterrupt: 